# Inspect the results of the assessment of a classifier's fairness based on movement patterns

In [ ]:
import pandas as pd
import numpy as np
import pickle

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates to retrive info necessary to plot on the grids.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)


# Read the results of an assessment from disk.
path_results = './res_exp.pkl'
with open(path_results, "rb") as f:
    dict_res = pickle.load(f)


# Read the geodataframes of the grids.
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

In [ ]:
# For each candidate, retrieve the grid and subset of cell it refers to.

# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Compute the number of objects associated with each candidate
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Retrieve the log-likelihood ratios computed for the candidates.
vec_lr_dataset = dict_res['vec_LR_dataset']
vec_inrate_dataset = dict_res['vec_inrate_dataset']
vec_outrate_dataset = dict_res['vec_outrate_dataset']
# display(vec_lr_dataset)

# For each candidate, here represented as a tuple of cell IDs, associate the grid and subset of cells it refers to.
num_candidates = vec_lr_dataset.size
list_grid_ids = np.empty(num_candidates, dtype=object)
list_cellids = np.empty(num_candidates, dtype=object)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=object)
    grid_id[0] = (int(grid[1]), int(grid[2]))
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    list_grid_ids[count : count + num_els_grid] = grid_id
    list_cellids[count : count + num_els_grid] = cell_ids

    count += num_els_grid

df_candidates = pd.DataFrame({
    "grid_id":   list_grid_ids,
    "cell_ids":  list_cellids,
    "lr":        vec_lr_dataset,
    "num_objs":  num_objs_candidates,
    "in_rate":   vec_inrate_dataset,
    "out_rate":  vec_outrate_dataset,
    # "num_cells": num_cells_candidates,   # <- add if you actually have it
})
display(df_candidates)

In [ ]:
target_grid = (1000, 0)
target_numcells_candidate = 1

# Select the candidates belonging to the 'target' grid.
df_sel_candidates = df_candidates.loc[df_candidates['grid_id'] == target_grid].copy()
# display(df_sel_candidates)

# Count the number of cells making up each selected candidate.
df_sel_candidates['num_cells'] = [len(subset) if isinstance(subset, tuple) else 1 for subset in df_sel_candidates['cell_ids']]
# display(df_sel_candidates)

# Select the candidates with the desired number of cells.
df_sel_candidates = df_sel_candidates.loc[df_sel_candidates['num_cells'] == target_numcells_candidate]
display(df_sel_candidates)

In [ ]:
#display(sel_cell_ids)
#display(sel_lr)

# Retrieve the geopandas dataframe of the grid of interest.
grid = dict_grids[target_grid].grid
grid.loc[:, ['lr', 'in', 'out']], grid.loc[:, 'nobjs'] = 0., 0

# Augment the dataframe with various info.
sel_cell_ids = df_sel_candidates['cell_ids']
grid.loc[sel_cell_ids, 'lr'] = df_sel_candidates['lr']
grid.loc[sel_cell_ids, 'nobjs'] = df_sel_candidates['num_objs']
grid.loc[sel_cell_ids, 'in'] = df_sel_candidates['in_rate']
grid.loc[sel_cell_ids, 'out'] = df_sel_candidates['out_rate']
grid.head(20)

In [ ]:
# Center map on grid bounds
minx, miny, maxx, maxy = grid.total_bounds
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    control_scale=True
)

# Build a continuous colormap from lr values
grid["cell_id"] = grid.index.astype(str)
vmin = float(grid["in"].min())
vmax = float(grid["in"].max())
colormap = cm.linear.YlOrRd_09.scale(vmin, vmax)  # pick any palette you like
colormap.caption = "positive rate"
colormap.add_to(m)

def style_fn(feature):
    val = feature["properties"].get("in", None)
    return {
        "color": "blue",
        "weight": 1,
        "fillColor": colormap(float(val)),
        "fillOpacity": 0.7,
    }

folium.GeoJson(
    grid,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=["cell_id", "lr", "nobjs", "in", "out"], 
                                  aliases=["cell ID:", "lr:", "num_objs:", "pr_in:", "pr_out:"]),
).add_to(m)

m

In [ ]:
# Center map on grid bounds
minx, miny, maxx, maxy = grid.total_bounds
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    control_scale=True
)

# Build a continuous colormap from lr values
grid["cell_id"] = grid.index.astype(str)
vmin = float(grid["lr"].min())
vmax = float(grid["lr"].max())
colormap = cm.linear.YlOrRd_09.scale(vmin, vmax)  # pick any palette you like
colormap.caption = "log-likelihood ratio"
colormap.add_to(m)

def style_fn(feature):
    val = feature["properties"].get("lr", None)
    return {
        "color": "blue",
        "weight": 1,
        "fillColor": colormap(float(val)),
        "fillOpacity": 0.7,
    }

folium.GeoJson(
    grid,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=["cell_id", "lr", "nobjs", "in", "out"], 
                                  aliases=["cell ID:", "lr:", "num_objs:", "pr_in:", "pr_out:"]),
).add_to(m)

m